# 02 · 数据类与类型标注 Dataclasses & Typing

> **能力标签**：SH1（Python 编程能力）

## 目标 Objectives

完成本课后，你应该能够：

1. 用 `@dataclass` 装饰器快速定义数据持有类（data-holding class）。
2. 理解 `frozen=True` 的作用，创建不可变（immutable）对象。
3. 为字段设置默认值，包括使用 `field(default_factory=...)` 的场景。
4. 写清晰的**类型标注**（type hints），包括 `X | None`（Python 3.10+）与 `Optional[X]`。
5. 用 `from __future__ import annotations` 启用延迟求值，解决前向引用问题。
6. 在 `@dataclass` 中定义 `@property`，暴露只读计算属性。

> 对应能力：**SH1**


## 概念讲解 Concepts

### 为什么要有 `@dataclass`？

如果你有一个类只是用来**存数据**，传统写法需要大量样板代码：

```python
class Point:
    def __init__(self, x: float, y: float):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"Point(x={self.x}, y={self.y})"

    def __eq__(self, other):
        return self.x == other.x and self.y == other.y
```

`@dataclass` 自动生成 `__init__`、`__repr__`、`__eq__`（以及可选的 `__hash__`、`__lt__` 等），让代码简洁很多。

---

### 基本用法

```python
from dataclasses import dataclass

@dataclass
class Point:
    x: float
    y: float
```

这三行等价于上面十几行手写代码！

---

### `frozen=True`：不可变对象

```python
@dataclass(frozen=True)
class Point:
    x: float
    y: float
```

- 赋值后字段**不可修改**（尝试修改会抛 `FrozenInstanceError`）。
- `frozen=True` 同时生成 `__hash__`，使对象可以作为字典键或加入集合。
- 适合表示数学上的"值"（value），如坐标、颜色、日期。

---

### 默认值与 `field`

```python
from dataclasses import dataclass, field

@dataclass
class Config:
    lr: float = 0.001
    tags: list = field(default_factory=list)   # 不要用 tags: list = [] ！
```

> **陷阱**：可变默认值（如 `[]`、`{}`）不能直接写，必须用 `field(default_factory=...)`，否则所有实例会共享同一个对象。

---

### 类型标注（Type Hints）

Python 本身不强制类型，但标注让代码更可读、工具（mypy、IDE）更智能：

```python
def greet(name: str, repeat: int = 1) -> str:
    return (name + " ") * repeat
```

常见类型：`int`、`float`、`str`、`bool`、`list[int]`、`dict[str, float]`、`tuple[int, ...]`

**`X | None`（Python 3.10+）**：表示"X 或 None"，等价于旧写法 `Optional[X]`：

```python
def find(key: str) -> str | None:
    ...
```

---

### `from __future__ import annotations`

在文件顶部加这一行，可以在类型标注中使用**前向引用**（即引用还未定义的类名）：

```python
from __future__ import annotations

@dataclass
class Node:
    value: int
    next: Node | None = None   # Node 还没定义完，但因为延迟求值所以没问题
```

同时也让 `@dataclass` 在 Python 3.9 上支持 `list[int]` 语法（不用写 `List[int]`）。

---

### `@property`：只读计算属性

```python
@dataclass(frozen=True)
class Point:
    x: float
    y: float

    @property
    def norm(self) -> float:
        import math
        return math.hypot(self.x, self.y)
```

调用时像访问属性一样：`p.norm`，而不是 `p.norm()`。
`frozen=True` 与 `@property` 搭配：字段不可改，但计算属性可以读。


## 示例 Worked Example

下面是 `w1-dataclass` 练习包里的 `Point` 数据类，完全自包含。


In [ ]:
from __future__ import annotations
from dataclasses import dataclass
import math


@dataclass(frozen=True)
class Point:
    """二维坐标点（不可变）。"""
    x: float
    y: float

    def dist_to(self, other: Point) -> float:
        """到另一个点的欧氏距离。"""
        return math.hypot(self.x - other.x, self.y - other.y)

    @property
    def norm(self) -> float:
        """到原点的距离（L2 范数）。"""
        return math.hypot(self.x, self.y)


# ── 验证 ──────────────────────────────────────────────────────────────────
p1 = Point(3.0, 4.0)
p2 = Point(0.0, 0.0)

print(f"p1 = {p1}")
print(f"p1.norm = {p1.norm}")          # 5.0
print(f"p1.dist_to(p2) = {p1.dist_to(p2)}")   # 5.0
print(f"p1 == Point(3, 4): {p1 == Point(3.0, 4.0)}")  # True（__eq__ 自动生成）

# frozen 验证
try:
    p1.x = 10
except Exception as e:
    print(f"frozen 保护生效：{type(e).__name__}")


In [ ]:
# 演示 default_factory 与可选类型标注
from __future__ import annotations
from dataclasses import dataclass, field


@dataclass
class Cluster:
    """一组点的集合。"""
    label: str
    points: list[Point] = field(default_factory=list)
    description: str | None = None

    def centroid(self) -> Point | None:
        """返回质心；若无点则返回 None。"""
        if not self.points:
            return None
        cx = sum(pt.x for pt in self.points) / len(self.points)
        cy = sum(pt.y for pt in self.points) / len(self.points)
        return Point(cx, cy)


cluster = Cluster(label="A", points=[Point(1, 2), Point(3, 4), Point(5, 6)])
print(cluster)
print(f"centroid = {cluster.centroid()}")
print(f"description is None: {cluster.description is None}")


## 动手 Exercise

在下面的 cell 中，**为 `Point` 添加 `manhattan` 属性**：

- Manhattan 距离（曼哈顿距离）到原点 = `|x| + |y|`
- 添加为 `@property`，返回 `float`
- 确保 `Point(3, -4).manhattan == 7.0`

> 提示：`@dataclass(frozen=True)` 里可以定义任意 `@property`，只是不能给字段赋值。


In [ ]:
from __future__ import annotations
from dataclasses import dataclass
import math


@dataclass(frozen=True)
class PointEx:
    """扩展版 Point，带 manhattan 属性。"""
    x: float
    y: float

    @property
    def norm(self) -> float:
        return math.hypot(self.x, self.y)

    @property
    def manhattan(self) -> float:
        # TODO: 实现 manhattan 距离到原点
        ...


In [ ]:
# 验证
try:
    pt = PointEx(3.0, -4.0)
    print("manhattan == 7.0 ?", pt.manhattan == 7.0)
    print("norm == 5.0 ?", abs(pt.norm - 5.0) < 1e-9)
except Exception as e:
    print("出错：", e)


## 延伸阅读 Further Reading

1. **Python 官方文档 — dataclasses 模块**：<https://docs.python.org/3/library/dataclasses.html>
2. **PEP 557 — Data Classes**（`@dataclass` 的设计提案）：<https://peps.python.org/pep-0557/>
3. **PEP 604 — Allow writing union types as `X | Y`**：<https://peps.python.org/pep-0604/>
4. **mypy 官方文档**（静态类型检查工具）：<https://mypy.readthedocs.io/>
5. **`attrs` 库**：`@dataclass` 的前身，功能更丰富，可以作为进阶参考。


## 关联练习 Related Assignments

本课对应练习包：本章 `assignments/`

**运行自动评分器**：

```bash
cd 本章 `assignments/`
python -m pytest test_hidden.py -v
```

或使用平台工具：

```bash
python check.py 02-dataclass
```

> 目标通过率 ≥ 70%（`threshold_pass: 0.7`）
> 能力标签：**SH1**
